# lmLite: browser-local inference

Run cells in order. This introduction loads only DistilGPT-2 and MiniLM, reuses them, and disposes them at the end. Defaults use WASM, not WebGPU. First use downloads weights unless hosted or cached; browser memory is still required. DistilGPT-2 is a completion model, not a reliable instruction-following assistant.

Optional experiments:
- [GPT-2](generation-gpt2.ipynb)
- [DistilGPT-2](generation-distilgpt2.ipynb)
- [TinyLlama](generation-tinyllama.ipynb)
- [LaMini-Neo](generation-lamini.ipynb)
- [Embeddings](embeddings.ipynb)
- [Classification](classification.ipynb)
- [Task pipelines](tasks.ipynb)
- [Model info and cache status](model-info.ipynb)
- [lmLite orchestration](orchestration.ipynb)

Use a fresh kernel between heavyweight trials. Disposal releases inference resources without deleting cached files; browser memory reclamation may be delayed.

In [ ]:
from lmlite import LLM

In [ ]:
llm = await LLM.create(
    generator_model="distilgpt2",
    embedding_model="all-MiniLM-L6-v2",
    backend="wasm",
    embedding_backend="wasm",
    embedding_dtype="q8",
    max_new_tokens=30,
    return_full_text=False,  # Return only the continuation, not the input prompt.
    do_sample=False,
)

In [ ]:
await llm.generate("Python is useful because")

In [ ]:
vector = await llm.embed("JupyterLite runs in the browser.")
print(f"[{str(vector[:3])[1:-1]}, ..., {vector[-1]}] ({len(vector)} dimensions)")

In [ ]:
docs = [
    "JupyterLite runs entirely in the browser.",
    "Python is widely used for machine learning.",
    "TypeScript is useful for frontend applications.",
]
query = "Where does JupyterLite run?"
results = await llm.similarity_search(query, docs, k=2)
for doc, score in results:
    print(f"{score:.3f} -> {doc}")

In [ ]:
# A minimal retrieval-augmented prompt, not a factual-answer guarantee.
context = "\n".join(doc for doc, _ in results)
prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
completion = await llm.generate(prompt)
print(completion.strip())

In [ ]:
await llm.dispose()

The instance is now disposed. Rerun the workflow cell to load again from cached files. Embeddings use mean pooling and L2 normalization; `embedding_dtype` and `embedding_backend` are independent of generation settings. See the dedicated notebooks for comparisons, classification, export, and chat examples.